# Genus three

Compute $S_q(n)$ for $q=5,7$, with weight $1/\#\operatorname{Aut}_{\mathbf F_q}(C)$. Multiplication by $\#\mathrm{GL}_3(\mathbf F_q)$ gives the smooth quartic codeword counts.

Curves with rational points are enumerated below. The zero-point terms use [Faber–Grantham–Howe, §8 and their accompanying data](https://github.com/RationalPoint/excessive). The complete $q=5$ distribution is checked against [Sutherland's database](https://math.mit.edu/~drew/avff/g3_spq_5.txt). Source hashes and the imported automorphism-order counts are in `data/smooth_references.json`.


Fix a rational point $P=(0:0:1)$ and its tangent $x=0$. We work in characteristic greater than three. Scaling the equation and shifting $z$ removes the $x^2z^2$ and $xyz^2$ terms, giving
$$
\begin{aligned}
F={}&xz^3+cy^2z^2+(dx^3+ex^2y+fxy^2+gy^3)z\\
&+hx^4+ix^3y+jx^2y^2+kxy^3+\ell y^4.
\end{aligned}
$$
On $x=0$, this becomes $y^2(cz^2+gyz+\ell y^2)$. The first nonzero coefficient among $c,g,\ell$ determines the tangent contact order $m$.

A shear $y\mapsto y+ux$, followed by removing the two $z^2$ terms again, changes $g$ by $-4c^2u/3$ for contact two, $f$ by $3gu$ for contact three, and $k$ by $4\ell u$ for contact four. These coefficients are nonzero, so we can choose $u$ to make the relevant term vanish. Scaling $x$ gives the forms used in the code.

| Contact at $P$ | Conditions | Remaining coordinate changes |
| --- | --- | --- |
| Two | $c=1,\ g=0$ | $(x,y,z)\mapsto(b^2x,by,z)$ |
| Three | $c=0,\ g=1,\ f=0$ | $(x,y,z)\mapsto(b^3x,by,z)$ |
| Four | $c=g=0,\ \ell=1,\ k=0$ | $(x,y,z)\mapsto(b^4x,by,z)$ |

The vanishing conditions fix the shifts and shear. The remaining changes, with $b\in\mathbf F_q^\times$, act by $F\mapsto b^{-m}F(b^m x,by,z)$. The values of $b$ that preserve a particular equation form $\operatorname{Aut}_{\mathbf F_q}(C,P)$. This accounts for all automorphisms fixing $P$, since smooth plane quartics have canonical plane embeddings.

`normal_forms` keeps one representative per coefficient orbit and records `orbit_size`. Its weight as a curve with a chosen point is $\texttt{orbit_size}/(q-1)=1/\#\operatorname{Aut}_{\mathbf F_q}(C,P)$.

To forget $P$, the code first uses points of contact two, if any exist. Otherwise it uses contact three, then contact four. Let $r$ be the number of rational points of the selected order. Automorphisms of $C$ can identify different choices of $P$. Summing over their orbits gives
$$
\sum_{[(C,P)]}\frac{1}{\#\operatorname{Aut}_{\mathbf F_q}(C,P)}
=\frac{r}{\#\operatorname{Aut}_{\mathbf F_q}(C)}.
$$
Dividing by $r$ gives the required weight of $C$. This is the update `orbit_size / ((q - 1) * marked)` in `smooth_distribution`, where `marked` is $r$.


The SageMath helper checks geometric smoothness using polynomial GCDs. In the chart $x=1$, depress the cubic in $z$ to $z^3+p(y)z+r(y)$. Triple-root singularities are detected by $\gcd(p,r,r')$. The remaining singularities are detected by $\gcd(4p^3+27r^2,\,2pr'-3rp')$, after removing all factors supported on $p=0$. The line $x=0$ is checked separately.

SageMath handles finite fields, polynomial arithmetic, derivatives, determinants, and projective points. The normal forms and counting weights are implemented in `src/smooth_quartics.py`.


In [1]:
from pathlib import Path
import json

root = Path('..').resolve()
load(str(root / 'src' / 'smooth_quartics.py'))
references = json.loads((root / 'data' / 'smooth_references.json').read_text())


Enumerate the three normal forms and add the classified pointless curves. The total weight is $q^6+1$. The $q=5$ comparison is made at every point count, using exact rational numbers.


In [2]:
distribution_results = {}
for q0 in [ZZ(5),ZZ(7)]:
    weights,candidates,smooth = smooth_distribution(q0)
    imported = references['pointless']['automorphism_order_counts'][str(q0)]
    weights[0] = sum(QQ(count)/ZZ(order) for order,count in imported.items())
    weights = dict(sorted(weights.items()))
    assert sum(weights.values()) == q0^6+1
    assert sum(n*weight for n,weight in weights.items()) == (q0+1)*(q0^6+1)
    if q0 == 5:
        expected = {ZZ(n):QQ(value) for n,value in references['q5_reference']['S'].items()}
        assert weights == expected
    factor = (q0-1)*q0^3*(q0^3-1)*(q0^2-1)
    coefficients = {n:factor*weight for n,weight in weights.items()}
    assert all(value >= 0 and value.denominator() == 1 for value in coefficients.values())
    distribution_results[q0] = (weights,coefficients,candidates,smooth)
    print('q =',q0)
    print(table([(n,weights[n],coefficients[n]) for n in weights],
                header_row=['rational points n','S_q(n)','smooth codewords']))
    print('Total weight =',sum(weights.values()))


q = 5
  rational points n   S_q(n)     smooth codewords
├───────────────────┼──────────┼──────────────────┤
  0                   135/8      25110000
  1                   297/2      220968000
  2                   4071/8     757206000
  3                   3673/3     1821808000
  4                   33319/16   3098667000
  5                   10949/4    4073028000
  6                   22367/8    4160262000
  7                   4807/2     3576408000
  8                   53917/32   2507140500
  9                   3152/3     1563392000
  10                  4367/8     812262000
  11                  543/2      403992000
  12                  653/6      161944000
  13                  153/4      56916000
  14                  61/8       11346000
  15                  2/3        992000
  16                  1/32       46500
Total weight = 15626
q = 7
  rational points n   S_q(n)      smooth codewords
├───────────────────┼───────────┼──────────────────┤
  0                   145/8      

Export both distributions for `weight_enumerator.ipynb`.


In [3]:
result_dir = root / 'results'
result_dir.mkdir(exist_ok=True)
payload = {
    'scope':'weighted smooth plane quartic counts from rational-point normal forms and classified pointless curves',
    'fields':{str(q0):{
        'S':{str(n):str(value) for n,value in values[0].items()},
        'coefficients':{str(n):str(value) for n,value in values[1].items()},
        'canonical_candidates':dict(zip(['contact two','contact three','contact four'],values[2].values())),
        'smooth_candidates':dict(zip(['contact two','contact three','contact four'],values[3].values())),
    } for q0,values in distribution_results.items()},
}
(result_dir / 'genus_three.json').write_text(json.dumps(payload,indent=2)+'\n')
print('Saved ../results/genus_three.json')


Saved ../results/genus_three.json
